# Transformer Model — Experiment Notebook
**Task 3: Transformer-based Time Series Forecasting**

In [ ]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.fetch_data import load_all_raw, STOCK_UNIVERSE
from src.data.preprocess import preprocess_all
from src.utils.metrics import evaluate
from src.models.transformer import (
    StockTransformer, train_transformer,
    predict_test_transformer, forecast_future_transformer,
    run_transformer_pipeline
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"PyTorch {torch.__version__}")

In [ ]:
raw = load_all_raw()
processed = preprocess_all(raw, save=False)
print(f"Loaded {len(processed)} stocks")

## Single-Stock Deep Dive (TCS.NS)

In [ ]:
TICKER = "TCS.NS"
SEQ_LEN = 60
EPOCHS = 30

d = processed[TICKER]
train, test = d["train"], d["test"]
tr_sc, te_sc, scaler = d["train_scaled"], d["test_scaled"], d["scaler"]

model = StockTransformer(seq_len=SEQ_LEN)
losses = train_transformer(model, tr_sc, seq_len=SEQ_LEN, epochs=EPOCHS)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color="darkcyan")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title(f"{TICKER} — Transformer Training Loss")
plt.tight_layout()
plt.show()

In [ ]:
raw_preds = predict_test_transformer(model, tr_sc, te_sc, scaler, SEQ_LEN)
pred = pd.Series(raw_preds, index=test.index, name="Transformer_Pred")

metrics = evaluate(test.values, pred.values, "Transformer", TICKER)
print(f"Transformer Metrics for {TICKER}:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-100:], label="Train", color="steelblue", alpha=0.5)
ax.plot(test, label="Actual", color="green", linewidth=2)
ax.plot(pred, label="Transformer", color="darkcyan", linewidth=1.5, linestyle="--")
ax.set_title(f"{TICKER} — Transformer Forecast vs Actual")
ax.legend()
plt.tight_layout()
plt.show()

## Run on All Stocks

In [ ]:
trans_preds, trans_fc, trans_met = run_transformer_pipeline(processed, epochs=EPOCHS)

trans_df = pd.DataFrame(trans_met)
print("\n── Transformer Results ──")
print(trans_df.to_string())
print(f"\nAvg MAPE: {trans_df['MAPE'].mean():.2f}%  |  Avg DirAcc: {trans_df['DirAcc'].mean():.1f}%")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()
for i, (ticker, pred) in enumerate(trans_preds.items()):
    if i >= 9: break
    ax = axes[i]
    actual = processed[ticker]["test"]
    ax.plot(actual, label="Actual", color="green", linewidth=1.5)
    ax.plot(pred, label="Transformer", color="darkcyan", linewidth=1, linestyle="--")
    ax.set_title(STOCK_UNIVERSE.get(ticker, ticker), fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.legend(fontsize=7)
plt.suptitle("Transformer Forecast vs Actual — All Stocks", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()